In [3]:
import pandas as pd
import math
from collections import Counter
from pprint import pprint

# Create the training dataset
data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain',
                'Rain', 'Rain', 'Overcast', 'Sunny',
                'Sunny', 'Rain', 'Sunny', 'Overcast',
                'Overcast', 'Rain'],
    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild',
                    'Cool', 'Cool', 'Cool', 'Mild',
                    'Cool', 'Mild', 'Mild', 'Mild',
                    'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High',
                 'Normal', 'Normal', 'Normal', 'High',
                 'Normal', 'Normal', 'Normal', 'High',
                 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak',
             'Weak', 'Strong', 'Strong', 'Weak',
             'Weak', 'Weak', 'Strong', 'Strong',
             'Weak', 'Strong'],
    'Play': ['No', 'No', 'Yes', 'Yes',
             'Yes', 'No', 'Yes', 'No',
             'Yes', 'Yes', 'Yes', 'Yes',
             'Yes', 'No']
}

# Save and read the CSV file
pd.DataFrame(data).to_csv('play_tennis.csv', index=False)
df = pd.read_csv('play_tennis.csv')

print("Training Dataset:")
print(df.to_string(index=False))


# Calculate entropy
def entropy(labels):
    counts = Counter(labels)
    total = len(labels)

    return -sum(
        (count / total) * math.log2(count / total)
        for count in counts.values()
    )


# Calculate information gain
def information_gain(df, attribute, target):
    total_entropy = entropy(df[target])
    weighted_entropy = 0

    for value in df[attribute].unique():
        subset = df[df[attribute] == value]
        weighted_entropy += (
            len(subset) / len(df)
        ) * entropy(subset[target])

    return total_entropy - weighted_entropy


# Build the decision tree using ID3
def id3(df, attributes, target):
    if len(df[target].unique()) == 1:
        return df[target].iloc[0]

    if not attributes:
        return df[target].mode()[0]

    gains = {
        attr: information_gain(df, attr, target)
        for attr in attributes
    }

    best = max(gains, key=gains.get)
    tree = {best: {}}

    for value in df[best].unique():
        subset = df[df[best] == value]

        if subset.empty:
            tree[best][value] = df[target].mode()[0]
        else:
            remaining = [a for a in attributes if a != best]
            tree[best][value] = id3(subset, remaining, target)

    return tree


# Classify a new sample
def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree

    attribute = next(iter(tree))
    value = sample.get(attribute)

    if value not in tree[attribute]:
        return "Unknown"

    return predict(tree[attribute][value], sample)


# Main program
attributes = list(df.columns[:-1])
target = 'Play'

print("\nInformation Gain:")
for attr in attributes:
    print(attr, ":", round(information_gain(df, attr, target), 4))

tree = id3(df, attributes, target)

print("\nDecision Tree:")
pprint(tree)

# Classify a new sample
sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

print("\nNew Sample:")
print(sample)

result = predict(tree, sample)
print("Classification Result:", result)

Training Dataset:
 Outlook Temperature Humidity   Wind Play
   Sunny         Hot     High   Weak   No
   Sunny         Hot     High Strong   No
Overcast         Hot     High   Weak  Yes
    Rain        Mild     High   Weak  Yes
    Rain        Cool   Normal   Weak  Yes
    Rain        Cool   Normal Strong   No
Overcast        Cool   Normal Strong  Yes
   Sunny        Mild     High   Weak   No
   Sunny        Cool   Normal   Weak  Yes
    Rain        Mild   Normal   Weak  Yes
   Sunny        Mild   Normal Strong  Yes
Overcast        Mild     High Strong  Yes
Overcast         Hot   Normal   Weak  Yes
    Rain        Mild     High Strong   No

Information Gain:
Outlook : 0.2467
Temperature : 0.0292
Humidity : 0.1518
Wind : 0.0481

Decision Tree:
{'Outlook': {'Overcast': 'Yes',
             'Rain': {'Wind': {'Strong': 'No', 'Weak': 'Yes'}},
             'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}}}

New Sample:
{'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Win